In [1]:
import pandas as pd

orders       = pd.read_csv("https://cdn.enqurious.com/documents/0518fd79-992c-420c-8580-a7acf31172b6_exorders.csv", parse_dates=["order_purchase_date"])
transactions = pd.read_csv("https://cdn.enqurious.com/documents/c1b6bff1-a6af-431b-bcc8-9618990bc5df_extransactions.csv")
products     = pd.read_csv("https://cdn.enqurious.com/documents/604aa913-c904-4e6c-9d26-fdd443c52482_exproducts.csv")

# Step 1: Merge orders → transactions → products
# (check actual column name casing with df.columns before merging)

# Step 2: Extract month_num, month_name, season

# Step 3: Aggregate revenue by month × season × category

## Unlocking Seasonal Trends at GlobalMart

**Full SQL mental model — understand the pipeline before writing Python:**

```sql
-- Step 1: JOIN the three tables
SELECT o.order_id, o.order_purchase_date, t.Sales_Amount, p.category
FROM orders o
LEFT JOIN transactions t ON o.order_id = t.Order_ID
LEFT JOIN products p ON t.Product_ID = p.product_id;

-- Step 2: Extract month + assign season
SELECT *, EXTRACT(MONTH FROM order_purchase_date) AS month_num,
  CASE WHEN EXTRACT(MONTH ...) IN (12,1,2) THEN 'Winter'
       WHEN EXTRACT(MONTH ...) IN (3,4,5)  THEN 'Spring'
       WHEN EXTRACT(MONTH ...) IN (6,7,8)  THEN 'Summer'
       ELSE 'Autumn' END AS season FROM joined;

-- Step 3: Aggregate monthly revenue per category
SELECT month_num, season, category, SUM(Sales_Amount) AS revenue
FROM enriched GROUP BY month_num, season, category ORDER BY month_num;

-- Step 4: Pivot (SQL is verbose — one CASE per category)
SELECT month_num, SUM(CASE WHEN category='Furniture' THEN revenue ELSE 0 END) AS Furniture, ...
FROM monthly GROUP BY month_num;   -- Python: pivot_table() does this in ONE line

-- Step 5: Melt (SQL needs UNION ALL for each column)
SELECT month_num, 'Furniture' AS category, Furniture AS revenue FROM wide UNION ALL ...
-- Python: .melt() does this in ONE line

-- Step 6: Rank within season
SELECT season, category, SUM(revenue) AS seasonal_revenue,
  DENSE_RANK() OVER (PARTITION BY season ORDER BY SUM(revenue) DESC) AS category_rank
FROM melted GROUP BY season, category;
```

**Key column name:** Revenue is in `Sales_Amount` (capital S, capital A) in the transactions table.

import pandas as pd

# ── Load required tables ─────────────────────────────────────────────────────
# parse_dates ensures order_purchase_date arrives as datetime, not plain string
# Without this, .dt.month would raise AttributeError — the column would be object type
orders = pd.read_csv(
    "https://cdn.enqurious.com/documents/0518fd79-992c-420c-8580-a7acf31172b6_exorders.csv",
    parse_dates=[
        "order_purchase_date", "order_approved_at",
        "order_dispatched_date", "order_delivered_date",
        "order_estimated_delivery_date"
    ]
)
transactions = pd.read_csv("https://cdn.enqurious.com/documents/c1b6bff1-a6af-431b-bcc8-9618990bc5df_extransactions.csv")
products     = pd.read_csv("https://cdn.enqurious.com/documents/604aa913-c904-4e6c-9d26-fdd443c52482_exproducts.csv")

# ── Step 1: Merge Orders → Transactions (SQL: LEFT JOIN on order_id) ─────────
# NOTE: column casing matters — orders uses 'order_id', transactions uses 'Order_ID'
# left_on/right_on handles the mismatch; the result keeps both columns (use df.columns to verify)
orders_txn = orders.merge(
    transactions,
    left_on="order_id",
    right_on="Order_ID",
    how="left"                # keep every order even if it has no matching transaction
)

# ── Step 2: Merge → Products (SQL: LEFT JOIN on product_id) ──────────────────
# Same casing mismatch: transactions uses 'Product_ID', products uses 'product_id'
df = orders_txn.merge(
    products,
    left_on="Product_ID",
    right_on="product_id",
    how="left"
)

# ── Step 3: Extract Time Components ──────────────────────────────────────────
# .dt is the pandas datetime accessor — works because parse_dates=True in read_csv above
# SQL equivalent: EXTRACT(MONTH FROM order_purchase_date)
df["month_num"]  = df["order_purchase_date"].dt.month
df["month_name"] = df["order_purchase_date"].dt.month_name()   # e.g. 'January'

# SQL equivalent: CASE WHEN month_num IN (12,1,2) THEN 'Winter' WHEN ... END
# Python: dict.map() — faster than apply() for lookup, works like a JOIN against a lookup table
season_map = {
    12: "Winter", 1: "Winter",  2: "Winter",
     3: "Spring", 4: "Spring",  5: "Spring",
     6: "Summer", 7: "Summer",  8: "Summer",
     9: "Autumn", 10: "Autumn", 11: "Autumn"
}
df["season"] = df["month_num"].map(season_map)

# ── Step 4: Aggregate Revenue by Month × Season × Category ───────────────────
# SQL: SELECT month_num, month_name, season, category, SUM(Sales_Amount) AS revenue
#      FROM df GROUP BY month_num, month_name, season, category ORDER BY month_num
# Named aggregation syntax: agg(new_col_name=('source_col', 'function'))
# IMPORTANT: column is 'Sales_Amount' — capital S and A — in the transactions CSV
base = (
    df
    .groupby(["month_num", "month_name", "season", "category"], as_index=False)
    .agg(revenue=("Sales_Amount", "sum"))
    .sort_values("month_num")
    .reset_index(drop=True)
)

print(base.head(12).to_string(index=False))

# ── Task 2: Pivot the base table — categories become columns ─────────────────
# SQL would need one SUM(CASE WHEN category='X' THEN revenue ELSE 0 END) per category.
# pandas pivot_table() auto-detects all unique values in `columns=` and creates one column each.
pivot_df = (
    base
    .pivot_table(
        index      = ["month_num", "month_name", "season"],  # one row per month
        columns    = "category",    # each unique category value becomes a column header
        values     = "revenue",     # cell values: total revenue for that month × category
        aggfunc    = "sum",         # aggregate if there are any duplicates in the index
        fill_value = 0              # NaN → 0 (no sales in that month doesn't mean missing data)
    )
    .reset_index()                  # promote month_num, month_name, season from index → columns
    .sort_values("month_num")
    .reset_index(drop=True)
)

# pivot_table() adds "category" as the column axis name — remove it for a clean header row
pivot_df.columns.name = None

print(pivot_df)

# ── Task 3: Melt the pivot table back to long format ─────────────────────────
# SQL would need UNION ALL for each category column — one SELECT per column.
# melt() does the reverse of pivot_table(): wide format → long format.
# id_vars  = columns to keep as row identifiers (not unpivoted)
# var_name = name for the new column that holds the old column headers
# value_name = name for the new column that holds the cell values
melted_df = (
    pivot_df
    .melt(
        id_vars    = ["month_num", "month_name", "season"],
        var_name   = "category",   # 'Furniture', 'Office Supplies', 'Technology' → column values
        value_name = "revenue"     # each cell's numeric value goes here
    )
    .sort_values(["month_num", "category"])
    .reset_index(drop=True)
)

print(melted_df.head(15))

# ── Step 4a: Aggregate to season level ───────────────────────────────────────
# SQL: SELECT season, category, SUM(revenue) AS seasonal_revenue FROM melted_df GROUP BY season, category
seasonal = (
    melted_df
    .groupby(["season", "category"], as_index=False)
    .agg(seasonal_revenue=("revenue", "sum"))
)

# ── Step 4b: Rank categories within each season ───────────────────────────────
# SQL: DENSE_RANK() OVER (PARTITION BY season ORDER BY seasonal_revenue DESC)
# groupby("season") = PARTITION BY season
# .rank(method="dense", ascending=False) = DENSE_RANK() ORDER BY ... DESC
# ascending=False → highest revenue gets rank 1
seasonal["category_rank"] = (
    seasonal
    .groupby("season")["seasonal_revenue"]
    .rank(method="dense", ascending=False)
    .astype(int)   # rank() returns float by default — cast to int for cleaner display
)

# ── Step 4c: Sort by season in calendar order (not alphabetical) ──────────────
# Alphabetical order would be: Autumn, Spring, Summer, Winter — not meaningful.
# Map each season to an integer, sort by that, then drop the helper column.
season_order = {"Spring": 1, "Summer": 2, "Autumn": 3, "Winter": 4}
seasonal["season_sort"] = seasonal["season"].map(season_order)

seasonal = (
    seasonal
    .sort_values(["season_sort", "category_rank"])
    .drop(columns=["season_sort"])   # helper column — no longer needed after sort
    .reset_index(drop=True)
)

print(seasonal)

In [5]:
import pandas as pd

#  Load required tables ──────────────────────────────────────────────────────
orders = pd.read_csv(
    "https://cdn.enqurious.com/documents/0518fd79-992c-420c-8580-a7acf31172b6_exorders.csv",
    parse_dates=[
        "order_purchase_date", "order_approved_at",
        "order_dispatched_date", "order_delivered_date",
        "order_estimated_delivery_date"
    ]
)
transactions = pd.read_csv("https://cdn.enqurious.com/documents/c1b6bff1-a6af-431b-bcc8-9618990bc5df_extransactions.csv")
products     = pd.read_csv("https://cdn.enqurious.com/documents/604aa913-c904-4e6c-9d26-fdd443c52482_exproducts.csv")

#  Step 1: Merge Orders → Transactions ──────────────────────────────────────
# NOTE: transactions table may use Order_ID / Product_ID casing
orders_txn = orders.merge(
    transactions,
    left_on="order_id",
    right_on="Order_ID",
    how="left"
)

# Step 2: Merge → Products ──────────────────────────────────────────────────
df = orders_txn.merge(
    products,
    left_on="Product_ID",
    right_on="product_id",
    how="left"
)

# Step 3: Extract Time Components ──────────────────────────────────────────
df["month_num"]  = df["order_purchase_date"].dt.month
df["month_name"] = df["order_purchase_date"].dt.month_name()

season_map = {
    12: "Winter", 1: "Winter",  2: "Winter",
     3: "Spring", 4: "Spring",  5: "Spring",
     6: "Summer", 7: "Summer",  8: "Summer",
     9: "Autumn", 10: "Autumn", 11: "Autumn"
}
df["season"] = df["month_num"].map(season_map)

# Step 4: Aggregate Revenue by Month × Season × Category ───────────────────
base = (
    df
    .groupby(["month_num", "month_name", "season", "category"], as_index=False)
    .agg(revenue=("Sales_Amount", "sum"))
    .sort_values("month_num")
    .reset_index(drop=True)
)

print(base.head(12).to_string(index=False))

 month_num month_name season        category     revenue
         1    January Winter       Furniture  62695.5571
         1    January Winter Office Supplies  58986.6470
         1    January Winter      Technology  53778.6280
         2   February Winter       Furniture  58640.1918
         2   February Winter Office Supplies  64841.3870
         2   February Winter      Technology  73089.5460
         3      March Spring       Furniture  73210.6145
         3      March Spring Office Supplies  86406.2450
         3      March Spring      Technology 100605.1070
         4      April Spring       Furniture  72294.2453
         4      April Spring Office Supplies  65120.9210
         4      April Spring      Technology  85366.5960


In [6]:
# ── Task 2: Pivot the base table — categories become columns ──────────────────
pivot_df = (
    base
    .pivot_table(
        index      = ["month_num", "month_name", "season"],  # one row per month
        columns    = "category",                              # one column per category
        values     = "revenue",                               # cell values: total revenue
        aggfunc    = "sum",                                   # aggregate if duplicates exist
        fill_value = 0                                        # replace NaN with 0
    )
    .reset_index()          # promote month_num, month_name, season back to columns
    .sort_values("month_num")
    .reset_index(drop=True)
)

# Remove the "category" axis label from the column header row
pivot_df.columns.name = None

print(pivot_df)

    month_num month_name  season   Furniture  Office Supplies  Technology
0           1    January  Winter  62695.5571        58986.647   53778.628
1           2   February  Winter  58640.1918        64841.387   73089.546
2           3      March  Spring  73210.6145        86406.245  100605.107
3           4      April  Spring  72294.2453        65120.921   85366.596
4           5        May  Spring  74611.9410        80913.812  100645.381
5           6       June  Summer  64588.0031        54742.978   81566.383
6           7       July  Summer  76340.7172        80089.575   79362.845
7           8     August  Summer  79228.3590        66515.831   68458.287
8           9  September  Autumn  25002.4226        27078.837   33462.389
9          10    October  Autumn  34243.0248        35618.063   31603.947
10         11   November  Autumn  63537.3576        61173.191   57231.810
11         12   December  Winter  45889.3128        29570.143   28586.170


In [7]:
# ── Task 3: Melt the pivot table back to long format ──────────────────────────
melted_df = (
    pivot_df
    .melt(
        id_vars    = ["month_num", "month_name", "season"],  # stay as row identifiers
        var_name   = "category",   # column headers → values in this new column
        value_name = "revenue"     # cell values → values in this new column
    )
    .sort_values(["month_num", "category"])
    .reset_index(drop=True)
)

print(melted_df.head(15))

    month_num month_name  season         category      revenue
0           1    January  Winter        Furniture   62695.5571
1           1    January  Winter  Office Supplies   58986.6470
2           1    January  Winter       Technology   53778.6280
3           2   February  Winter        Furniture   58640.1918
4           2   February  Winter  Office Supplies   64841.3870
5           2   February  Winter       Technology   73089.5460
6           3      March  Spring        Furniture   73210.6145
7           3      March  Spring  Office Supplies   86406.2450
8           3      March  Spring       Technology  100605.1070
9           4      April  Spring        Furniture   72294.2453
10          4      April  Spring  Office Supplies   65120.9210
11          4      April  Spring       Technology   85366.5960
12          5        May  Spring        Furniture   74611.9410
13          5        May  Spring  Office Supplies   80913.8120
14          5        May  Spring       Technology  1006

In [8]:
# ── Season-level revenue with within-season category rank ─────────────────────

# Step 1: Aggregate revenue by season and category
seasonal = (
    melted_df
    .groupby(["season", "category"], as_index=False)
    .agg(seasonal_revenue=("revenue", "sum"))
)

# Step 2: Rank categories within each season (rank 1 = highest revenue)
seasonal["category_rank"] = (
    seasonal
    .groupby("season")["seasonal_revenue"]
    .rank(method="dense", ascending=False)
    .astype(int)
)

# Step 3: Sort by season (custom calendar order) and rank
season_order = {"Spring": 1, "Summer": 2, "Autumn": 3, "Winter": 4}
seasonal["season_sort"] = seasonal["season"].map(season_order)

seasonal = (
    seasonal
    .sort_values(["season_sort", "category_rank"])
    .drop(columns=["season_sort"])
    .reset_index(drop=True)
)

print(seasonal)

    season         category  seasonal_revenue  category_rank
0   Spring       Technology       286617.0840              1
1   Spring  Office Supplies       232440.9780              2
2   Spring        Furniture       220116.8008              3
3   Summer       Technology       229387.5150              1
4   Summer        Furniture       220157.0793              2
5   Summer  Office Supplies       201348.3840              3
6   Autumn  Office Supplies       123870.0910              1
7   Autumn        Furniture       122782.8050              2
8   Autumn       Technology       122298.1460              3
9   Winter        Furniture       167225.0617              1
10  Winter       Technology       155454.3440              2
11  Winter  Office Supplies       153398.1770              3
